## Setup

In [2]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_URL = "https://pro.api.openmeasures.io"
jwt_token = os.getenv("OPEN_MEASURES_TOKEN")

_headers = {
    "Authorization": f"Bearer {jwt_token}"
}

In [3]:
# see quota
response = requests.get(f"{API_URL}/quota", headers=_headers)
data = response.json()

print(f"Status: {response.status_code}")
print(f"{data['organization_usage']['core_api_monthly_requests_count'] / data['monthly_limits']['core_api_monthly_request_limit'] * 100:.2f}% of monthly quota used")

Status: 200
1.83% of monthly quota used


## Sample Statistics

In [4]:
# Check and compare query sizes
from query import ANTI, ISRAEL, PALESTINE

a = ANTI
i = ISRAEL
p = PALESTINE

# dedupe is after sampling for fuzzy-identical cases
queries = {
    "ip_not": f'(({i}) OR ({p})) AND NOT ({a})',
    "ip_and": f'(({i}) OR ({p})) AND ({a})',
    "anti_only": f'({a}) AND NOT (({i}) OR ({p}))',
}

sites_to_test = ["4chan", "8kun", "truthsocial", "bluesky"]

results_summary = {}

for query_label, term_query in queries.items():
    results_summary[query_label] = {}
    for site in sites_to_test:
        test_params = {
            "sortdesc": "true",
            "limit": 1,
            "site": site,
            "term": term_query,
            "since": "2025-01-01",
            "until": "2025-12-31",
            "standard_fields": "true",
            "querytype": "boolean_content",
        }
        response = requests.get(f"{API_URL}/content", headers=_headers, params=test_params)
        print(f"====| {query_label} | {site} |====")
        print(f"Status: {response.status_code}")
        if response.status_code == 200:
            data = response.json()
            total_hits = data.get('total_hits')
            print(f"Total hits: {total_hits}")
            results_summary[query_label][site] = total_hits
        else:
            print(response.text[:300])
            results_summary[query_label][site] = None
        print()

print("\n=== SUMMARY ===")
for query_label, site_hits in results_summary.items():
    print(f"\n{query_label}:")
    for site, hits in site_hits.items():
        print(f"  {site}: {hits}")

====| ip_not | 4chan |====
Status: 200
Total hits: 887403

====| ip_not | 8kun |====
Status: 200
Total hits: 38070

====| ip_not | truthsocial |====
Status: 200
Total hits: 1148190

====| ip_not | bluesky |====
Status: 200
Total hits: 9783458

====| ip_and | 4chan |====
Status: 200
Total hits: 308888

====| ip_and | 8kun |====
Status: 200
Total hits: 9458

====| ip_and | truthsocial |====
Status: 200
Total hits: 132377

====| ip_and | bluesky |====
Status: 200
Total hits: 406729

====| anti_only | 4chan |====
Status: 200
Total hits: 2906601

====| anti_only | 8kun |====
Status: 200
Total hits: 30317

====| anti_only | truthsocial |====
Status: 200
Total hits: 576697

====| anti_only | bluesky |====
Status: 200
Total hits: 1376242


=== SUMMARY ===

ip_not:
  4chan: 887403
  8kun: 38070
  truthsocial: 1148190
  bluesky: 9783458

ip_and:
  4chan: 308888
  8kun: 9458
  truthsocial: 132377
  bluesky: 406729

anti_only:
  4chan: 2906601
  8kun: 30317
  truthsocial: 576697
  bluesky: 1376242

In [5]:
# Descriptive statistics

import pandas as pd

strata = ["ip_not", "ip_and", "anti_only"]

# results_summary structure: {query_label: {site: total_hits}}
df = pd.DataFrame(results_summary).T  # rows = query variant, columns = site
df.index.name = "query_variant"

# reconstruct 'full' as the sum of the three mutually exclusive partitions (sanity check + denominator)
df.loc["full"] = df.loc[strata].sum()

print("=== Raw hit counts per stratum ===")
print(df.to_string())

print("\n=== Composition: % of full corpus each stratum represents ===")
composition_df = (df.div(df.loc["full"], axis=1) * 100).round(2)
print(composition_df.to_string())

print("\n=== Composition by site ===")
for site in df.columns:
    print(f"\n{site}: total = {df.loc['full', site]:,}")
    for stratum in strata:
        pct = composition_df.loc[stratum, site]
        count = df.loc[stratum, site]
        print(f"  {stratum:<12} {count:>10,} ({pct:>5.2f}%)")

print("\n=== Cross-platform share: where does each stratum's volume come from? ===")
row_share_df = (df.div(df.sum(axis=1), axis=0) * 100).round(2)
print(row_share_df.loc[strata].to_string())

=== Raw hit counts per stratum ===
                 4chan   8kun  truthsocial   bluesky
query_variant                                       
ip_not          887403  38070      1148190   9783458
ip_and          308888   9458       132377    406729
anti_only      2906601  30317       576697   1376242
full           4102892  77845      1857264  11566429

=== Composition: % of full corpus each stratum represents ===
                4chan    8kun  truthsocial  bluesky
query_variant                                      
ip_not          21.63   48.90        61.82    84.58
ip_and           7.53   12.15         7.13     3.52
anti_only       70.84   38.95        31.05    11.90
full           100.00  100.00       100.00   100.00

=== Composition by site ===

4chan: total = 4,102,892
  ip_not          887,403 (21.63%)
  ip_and          308,888 ( 7.53%)
  anti_only     2,906,601 (70.84%)

8kun: total = 77,845
  ip_not           38,070 (48.90%)
  ip_and            9,458 (12.15%)
  anti_only        3

## Stratified Sampling

In [4]:
import os
import re
import json
import requests
import pandas as pd
from query import ANTI, ISRAEL, PALESTINE

a = ANTI
i = ISRAEL
p = PALESTINE

# dedupe is after sampling for fuzzy-identical cases
queries = {
    "ip_not": f'(({i}) OR ({p})) AND NOT ({a})',
    "ip_and": f'(({i}) OR ({p})) AND ({a})',
    "anti_only": f'({a}) AND NOT (({i}) OR ({p}))',
}

# max pages per stratum
PAGINATE_BY_QUERY = {
    "anti_only": 5,
    "ip_not": 3,
    "ip_and": 2,
}

platforms = ["4chan", "8kun", "truthsocial", "bluesky"]

months = [
    ("january", "2025-01-01", "2025-01-31"),
    ("february", "2025-02-01", "2025-02-28"),
    ("march", "2025-03-01", "2025-03-31"),
    ("april", "2025-04-01", "2025-04-30"),
    ("may", "2025-05-01", "2025-05-31"),
    ("june", "2025-06-01", "2025-06-30"),
    ("july", "2025-07-01", "2025-07-31"),
    ("august", "2025-08-01", "2025-08-31"),
    ("september", "2025-09-01", "2025-09-30"),
    ("october", "2025-10-01", "2025-10-31"),
    ("november", "2025-11-01", "2025-11-30"),
    ("december", "2025-12-01", "2025-12-31"),
]

DATA_DIR = "data"
CURSOR_INDEX_PATH = os.path.join(DATA_DIR, "cursor_index.json")

# -----------------------------------------------------------------------------
# SETUP: create data/ and one subfolder per platform if they don't exist
# -----------------------------------------------------------------------------

os.makedirs(DATA_DIR, exist_ok=True)
for platform in platforms:
    os.makedirs(os.path.join(DATA_DIR, platform), exist_ok=True)


# -----------------------------------------------------------------------------
# HELPER: cursor index persistence
# -----------------------------------------------------------------------------

def load_cursor_index():
    if os.path.exists(CURSOR_INDEX_PATH):
        with open(CURSOR_INDEX_PATH) as f:
            return json.load(f)
    return {}

def save_cursor(platform, query_label, month_name, page_num, cursor):
    index = load_cursor_index()
    key = f"{platform}__{query_label}__{month_name}__{page_num}"
    index[key] = cursor
    with open(CURSOR_INDEX_PATH, "w") as f:
        json.dump(index, f, indent=2)

def get_cursor(platform, query_label, month_name, page_num):
    index = load_cursor_index()
    key = f"{platform}__{query_label}__{month_name}__{page_num}"
    return index.get(key)


# -----------------------------------------------------------------------------
# HELPER: coerce mixed-type object columns to string for pyarrow compatibility
# -----------------------------------------------------------------------------

def clean_for_parquet(df):
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str)
    return df


# -----------------------------------------------------------------------------
# HELPER: check if a cell is already saved, recover parquet from csv if needed
# -----------------------------------------------------------------------------

def cell_already_saved(platform, query_label, page_num, month_name):
    base_name = f"{query_label}_{page_num}_{month_name}"
    csv_path = os.path.join(DATA_DIR, platform, f"{base_name}.csv")
    parquet_path = os.path.join(DATA_DIR, platform, f"{base_name}.parquet")
    json_path = os.path.join(DATA_DIR, platform, f"{base_name}.json")

    if os.path.exists(csv_path) and os.path.exists(parquet_path):
        return True

    # csv exists but parquet failed: reconstruct from csv, no re-query needed
    if os.path.exists(csv_path) and not os.path.exists(parquet_path):
        print(f"  recovering parquet from csv: {base_name}")
        try:
            df = pd.read_csv(csv_path)
            clean_for_parquet(df.copy()).to_parquet(parquet_path, engine="pyarrow", index=False)
            if os.path.exists(json_path):
                os.remove(json_path)
            print(f"  parquet recovered successfully")
        except Exception as e:
            print(f"  parquet recovery failed: {e}")
        return True

    # json temp file exists and csv write also failed: reconstruct both
    if os.path.exists(json_path):
        print(f"  recovering from temp json: {base_name}")
        try:
            with open(json_path) as f:
                results = json.load(f)
            df = pd.json_normalize(results)
            if "text" in df.columns:
                df["text"] = df["text"].fillna("").apply(
                    lambda t: re.sub(r'http\S+|www\.\S+', '[URL]', t)
                )
            df.to_csv(csv_path, index=False)
            clean_for_parquet(df.copy()).to_parquet(parquet_path, engine="pyarrow", index=False)
            os.remove(json_path)
            print(f"  both files recovered from temp json")
        except Exception as e:
            print(f"  json recovery failed: {e}")
        return True

    return False


# -----------------------------------------------------------------------------
# HELPER: save one page of results to both csv and parquet
# -----------------------------------------------------------------------------

def save_page(results, platform, query_label, page_num, month_name):
    if not results:
        print(f"  page {page_num}: 0 results, nothing to save")
        return 0

    df = pd.json_normalize(results)

    # strip URLs from text immediately on ingestion
    # [URL] token preserves the signal that a link was present.
    if "text" in df.columns:
        df["text"] = df["text"].fillna("").apply(
            lambda t: re.sub(r'http\S+|www\.\S+', '[URL]', t)
        )

    base_name = f"{query_label}_{page_num}_{month_name}"
    csv_path = os.path.join(DATA_DIR, platform, f"{base_name}.csv")
    parquet_path = os.path.join(DATA_DIR, platform, f"{base_name}.parquet")
    json_path = os.path.join(DATA_DIR, platform, f"{base_name}.json")

    # save csv first
    df.to_csv(csv_path, index=False)

    # temp save raw json in case parquet write fails
    with open(json_path, "w") as f:
        json.dump(results, f)

    try:
        clean_for_parquet(df.copy()).to_parquet(parquet_path, engine="pyarrow", index=False)
        os.remove(json_path)
    except Exception as e:
        print(f"  parquet write failed for {base_name}: {e}")
        print(f"  temp json saved at {json_path} for recovery")

    print(f"  page {page_num}: saved {len(df)} rows -> {csv_path} and {parquet_path}")
    return len(df)


# -----------------------------------------------------------------------------
# MAIN COLLECTION LOOP: month x platform x query x page
# -----------------------------------------------------------------------------

grand_total = 0
skipped = 0

for month_name, since_date, until_date in months:
    print(f"\n########## MONTH: {month_name} ##########")

    for platform in platforms:
        print(f"\n==== PLATFORM: {platform} ====")

        for query_label, term_query in queries.items():
            max_pages = PAGINATE_BY_QUERY[query_label]
            print(f"\n-- query: {query_label} (max {max_pages} pages) --")

            search_after_cursor = None
            page_num = 1

            while page_num <= max_pages:

                if cell_already_saved(platform, query_label, page_num, month_name):
                    print(f"  page {page_num}: already saved, skipping")
                    skipped += 1
                    search_after_cursor = get_cursor(platform, query_label, month_name, page_num)
                    page_num += 1
                    continue

                params = {
                    "sortdesc": "true",
                    "limit": 10000,
                    "site": platform,
                    "term": term_query,
                    "since": since_date,
                    "until": until_date,
                    "standard_fields": "true",
                    "querytype": "boolean_content",
                }
                if search_after_cursor:
                    params["search_after"] = search_after_cursor

                response = requests.get(f"{API_URL}/content", headers=_headers, params=params)

                if response.status_code != 200:
                    print(f"  page {page_num}: FAILED, status {response.status_code}")
                    print(f"  {response.text[:300]}")
                    break

                data = response.json()
                results = data.get("results", [])
                n_results = len(results)
                total_hits = data.get("total_hits")

                print(f"  page {page_num}: {n_results} results (total_hits: {total_hits})")

                n_saved = save_page(results, platform, query_label, page_num, month_name)
                grand_total += n_saved

                # early stop: a page with fewer than 10,000 results means
                # there's nothing left to paginate through for this cell
                if n_results < 10000:
                    print(f"  page {page_num} returned < 10,000 results, stopping early for this cell")
                    break

                search_after_cursor = data.get("search_after")
                if search_after_cursor:
                    save_cursor(platform, query_label, month_name, page_num, search_after_cursor)
                if not search_after_cursor:
                    print(f"  no further search_after cursor, stopping for this cell")
                    break

                page_num += 1

print(f"\n\nDONE. Total rows collected: {grand_total} new rows. Skipped {skipped} already-saved cells.")


########## MONTH: january ##########

==== PLATFORM: 4chan ====

-- query: ip_not (max 3 pages) --
  page 1: 10000 results (total_hits: 50844)
  page 1: saved 10000 rows -> data\4chan\ip_not_1_january.csv and data\4chan\ip_not_1_january.parquet
  page 2: 10000 results (total_hits: 50844)
  page 2: saved 10000 rows -> data\4chan\ip_not_2_january.csv and data\4chan\ip_not_2_january.parquet
  page 3: 10000 results (total_hits: 50844)
  page 3: saved 10000 rows -> data\4chan\ip_not_3_january.csv and data\4chan\ip_not_3_january.parquet

-- query: ip_and (max 2 pages) --
  page 1: 10000 results (total_hits: 19396)
  page 1: saved 10000 rows -> data\4chan\ip_and_1_january.csv and data\4chan\ip_and_1_january.parquet
  page 2: 9396 results (total_hits: 19396)
  page 2: saved 9396 rows -> data\4chan\ip_and_2_january.csv and data\4chan\ip_and_2_january.parquet
  page 2 returned < 10,000 results, stopping early for this cell

-- query: anti_only (max 5 pages) --
  page 1: 10000 results (total_hit

KeyboardInterrupt: 

## Dedupe 
Exact by ID and Fuzzy with Hashing  
Independent of platform so the models (platform agnostic) don't overfit

In [ ]:
import os
import glob
import hashlib
import pandas as pd

DATA_DIR = "data"
platforms = ["4chan", "8kun", "truthsocial", "bluesky"]

# -----------------------------------------------------------------------------
# LOAD: read all saved parquet files into one combined dataframe
# keeping platform and stratum as columns for post-hoc analysis
# -----------------------------------------------------------------------------

all_dfs = []
for platform in platforms:
    parquet_files = glob.glob(os.path.join(DATA_DIR, platform, "*.parquet"))
    for fpath in parquet_files:
        df = pd.read_parquet(fpath, engine="pyarrow")
        df["platform"] = platform
        fname = os.path.basename(fpath)
        df["stratum"] = fname.split("_")[0]
        df["source_file"] = fname
        all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"Combined before cleaning: {len(combined_df)} rows across {len(all_dfs)} files")

# save pre-dedup corpus
combined_df.to_parquet(os.path.join(DATA_DIR, "corpus_raw.parquet"), engine="pyarrow", index=False)
print(f"Saved raw corpus: {len(combined_df)} rows to corpus_raw.parquet")

# -----------------------------------------------------------------------------
# DEDUP 1: exact platform-native ID within platform
# cross-platform duplicates are preserved since platform is a key variable
# -----------------------------------------------------------------------------

def dedupe_by_id(df, id_col="id", platform_col="platform"):
    before = len(df)
    df = df.drop_duplicates(subset=[platform_col, id_col], keep="first")
    print(f"ID dedup: {before} -> {len(df)} rows ({before - len(df)} removed)")
    return df

# -----------------------------------------------------------------------------
# DEDUP 2: exact text hash within platform
# catches literal reposts with different IDs (bots, cross-posting)
# normalizes minimally: strip whitespace, lowercase
# does NOT dedupe across platforms
# -----------------------------------------------------------------------------

def dedupe_by_exact_text_hash(df, text_col="text", platform_col="platform"):
    before = len(df)
    df["_text_hash"] = df[text_col].fillna("").apply(
        lambda t: hashlib.sha256(t.strip().lower().encode("utf-8")).hexdigest()
    )
    df = df.drop_duplicates(subset=[platform_col, "_text_hash"], keep="first")
    df = df.drop(columns=["_text_hash"])
    print(f"Exact text dedup: {before} -> {len(df)} rows ({before - len(df)} removed)")
    return df

combined_df = dedupe_by_id(combined_df, id_col="id", platform_col="platform")
combined_df = dedupe_by_exact_text_hash(combined_df, text_col="text", platform_col="platform")

print(f"\nFinal deduped corpus: {len(combined_df)} rows")
print(combined_df.groupby(["platform", "stratum"]).size().reset_index(name="rows").to_string(index=False))

combined_df.to_parquet(os.path.join(DATA_DIR, "corpus_deduped.parquet"), engine="pyarrow", index=False)
print(f"\nSaved to {os.path.join(DATA_DIR, 'corpus_deduped.parquet')}")